# UTS Machine Learning - Linear & Logistic Regression
* **Nama**: Rivaldo Ferdiansyah
* **NIM**: 241011400456
* **Kelas**: Reg B
* **Random State**: 56

## Bagian A – Linear Regression

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import io

# Set random seed
RANDOM_STATE = 56
np.random.seed(RANDOM_STATE)

# Dataset Ads_Data
data_a_str = """BiayaIklan\tPenjualan
0.8\t34.3
0.9\t38.8
1.0\t33.2
1.3\t36.0
1.7\t51.3
1.9\t50.9
2.4\t50.0
2.5\t50.6
2.6\t52.9
2.7\t50.0
2.7\t52.8
2.9\t55.6
3.3\t68.0
3.4\t51.7
4.0\t70.9
4.3\t78.4
4.4\t62.9
4.7\t72.4
4.8\t73.8
4.9\t72.0
5.0\t70.9
5.1\t68.4
5.2\t61.5
5.3\t80.5
5.4\t56.4
5.8\t84.7
5.8\t87.3
6.0\t76.5
6.6\t92.4
7.0\t97.0
7.0\t97.1
7.1\t90.4
7.9\t103.0
8.9\t111.2
9.0\t105.6
9.2\t109.2
9.3\t117.8
9.7\t113.7
9.8\t121.1
9.9\t119.7"""

df_a = pd.read_csv(io.StringIO(data_a_str), sep='\t')
print("Jumlah baris Ads_Data:", len(df_a))
display(df_a.head())

X_a = df_a[['BiayaIklan']].values
y_a = df_a['Penjualan'].values

def train_test_split(X, y, test_size=0.2, random_state=None):
    if random_state is not None:
        np.random.seed(random_state)
    n = len(X)
    indices = np.arange(n)
    np.random.shuffle(indices)
    test_count = int(n * test_size)
    return X[indices[test_count:]], X[indices[:test_count]], y[indices[test_count:]], y[indices[:test_count]]

X_train_a, X_test_a, y_train_a, y_test_a = train_test_split(X_a, y_a, test_size=0.2, random_state=RANDOM_STATE)

# Scatter Plot
plt.figure(figsize=(8, 5))
plt.scatter(X_a, y_a, color='blue', label='Data')
plt.title('Scatter Plot Biaya Iklan vs Penjualan')
plt.xlabel('Biaya Iklan (juta Rp)')
plt.ylabel('Penjualan (juta Rp)')
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
class LinearRegressionNormal:
    def __init__(self):
        self.theta = None
        self.intercept = None
        self.coef = None
        
    def fit(self, X, y):
        X_b = np.c_[np.ones((X.shape[0], 1)), X]
        self.theta = np.linalg.inv(X_b.T.dot(X_b)).dot(X_b.T).dot(y)
        self.intercept = self.theta[0]
        self.coef = self.theta[1:]
        
    def predict(self, X):
        X_b = np.c_[np.ones((X.shape[0], 1)), X]
        return X_b.dot(self.theta)

lr_normal = LinearRegressionNormal()
lr_normal.fit(X_train_a, y_train_a)
print(f"Intercept: {lr_normal.intercept:.4f}")
print(f"Koefisien: {lr_normal.coef[0]:.4f}")
print(f"Persamaan: Penjualan = {lr_normal.intercept:.4f} + {lr_normal.coef[0]:.4f} * BiayaIklan")

def mae(y_true, y_pred): return np.mean(np.abs(y_true - y_pred))
def mse(y_true, y_pred): return np.mean((y_true - y_pred)**2)
def rmse(y_true, y_pred): return np.sqrt(mse(y_true, y_pred))
def r2(y_true, y_pred):
    return 1 - (np.sum((y_true - y_pred)**2) / np.sum((y_true - np.mean(y_true))**2))

y_pred_train = lr_normal.predict(X_train_a)
y_pred_test = lr_normal.predict(X_test_a)

metrics_df = pd.DataFrame({
    'Training': [mae(y_train_a, y_pred_train), mse(y_train_a, y_pred_train), rmse(y_train_a, y_pred_train), r2(y_train_a, y_pred_train)],
    'Testing': [mae(y_test_a, y_pred_test), mse(y_test_a, y_pred_test), rmse(y_test_a, y_pred_test), r2(y_test_a, y_pred_test)]
}, index=['MAE', 'MSE', 'RMSE', 'R2 Score'])
display(metrics_df)

**Jawaban A3 (Interpretasi)**:
Setiap kenaikan biaya iklan sebesar **1 juta Rupiah**, penjualan diprediksi akan meningkat sebesar **8.9851 juta Rupiah**, dengan asumsi faktor lain konstan (baseline penjualan tanpa iklan adalah 28.5405 juta Rupiah).

**Jawaban A4 (Analisis)**:
Nilai **RMSE** pada data testing sebesar **2.87** menunjukkan rata-rata besar kesalahan prediksi model terhadap data aktual. Nilai **$R^2$** sebesar **0.9873** pada data testing mengindikasikan bahwa sekitar **98.73%** variasi penjualan dapat dijelaskan oleh variabel biaya iklan. Model ini memiliki performa yang sangat baik dan kemampuan generalisasi yang tinggi.

In [ ]:
class LinearRegressionGD:
    def __init__(self, learning_rate=0.01, n_iterations=1000):
        self.lr = learning_rate
        self.n_iter = n_iterations
        self.theta = None
        self.loss_history = []
        
    def fit(self, X, y):
        m = X.shape[0]
        X_b = np.c_[np.ones((m, 1)), X]
        self.theta = np.zeros(X_b.shape[1])
        
        for i in range(self.n_iter):
            predictions = X_b.dot(self.theta)
            errors = predictions - y
            mse = np.mean(errors ** 2)
            self.loss_history.append(mse)
            
            gradients = (2/m) * X_b.T.dot(errors)
            self.theta -= self.lr * gradients
            
        self.intercept = self.theta[0]
        self.coef = self.theta[1:]

lrs = [0.001, 0.02, 0.1]
plt.figure(figsize=(9, 5))
for lr in lrs:
    gd = LinearRegressionGD(learning_rate=lr, n_iterations=5000)
    try:
        gd.fit(X_train_a, y_train_a)
        plt.plot(gd.loss_history, label=f'LR = {lr}')
        print(f"LR {lr} -> Intercept: {gd.intercept:.4f}, Coef: {gd.coef[0]:.4f}")
    except Exception as e:
        print(f"LR {lr} -> Gagal / Overflow: {e}")

plt.yscale('log')
plt.title('Grafik Loss (MSE) vs Iterasi')
plt.xlabel('Iterasi')
plt.ylabel('Loss (MSE - Log Scale)')
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
# Visualisasi Garis Regresi
plt.figure(figsize=(8, 5))
plt.scatter(X_test_a, y_test_a, color='red', label='Data Testing')
X_line = np.linspace(X_a.min(), X_a.max(), 100).reshape(-1, 1)
y_line = lr_normal.predict(X_line)
plt.plot(X_line, y_line, color='green', linewidth=2, label='Garis Regresi')
plt.title('Visualisasi Garis Regresi vs Data Testing')
plt.xlabel('Biaya Iklan (juta Rp)')
plt.ylabel('Penjualan (juta Rp)')
plt.grid(True)
plt.legend()
plt.show()

# Prediksi
for val in [4.5, 7.0, 12.0]:
    pred = lr_normal.predict(np.array([[val]]))
    print(f"Biaya Iklan {val} juta -> Prediksi Penjualan: {pred[0]:.4f} juta Rp")

## Bagian B – Logistic Regression

In [ ]:
# Dataset Email_Data
data_b_str = """JumlahLink\tKataPromo\tSpam
0\t0\t0
2\t4\t0
6\t7\t1
10\t8\t1
8\t7\t1
9\t11\t1
0\t6\t0
0\t8\t0
6\t5\t0
1\t9\t1
7\t3\t1
9\t3\t1
1\t8\t0
4\t2\t0
3\t6\t0
5\t9\t1
4\t4\t0
3\t12\t1
4\t5\t0
2\t1\t0
9\t8\t1
1\t7\t1
6\t0\t0
9\t9\t1
0\t3\t0
8\t4\t1
4\t11\t1
4\t3\t0
4\t2\t0
3\t10\t0
1\t3\t0
1\t9\t0
8\t2\t0
6\t6\t0
3\t11\t1
6\t9\t1
6\t9\t1
9\t1\t0
10\t7\t1
8\t1\t0
6\t6\t1
4\t6\t0
4\t9\t1
7\t4\t0
3\t10\t1
2\t4\t0
9\t5\t1
9\t12\t1
2\t5\t0
4\t7\t0
9\t1\t0
6\t0\t0
0\t5\t0
1\t4\t0
7\t10\t1
0\t11\t0
3\t9\t0
8\t10\t1
7\t1\t0
4\t7\t0"""

df_b = pd.read_csv(io.StringIO(data_b_str), sep='\t')
print("Jumlah email:", len(df_b))
print("Distribusi Spam:\n", df_b['Spam'].value_counts())

X_b = df_b[['JumlahLink', 'KataPromo']].values
y_b = df_b['Spam'].values

X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(X_b, y_b, test_size=0.25, random_state=RANDOM_STATE)

In [ ]:
class LogisticRegressionGD:
    def __init__(self, learning_rate=0.1, n_iterations=10000):
        self.lr = learning_rate
        self.n_iter = n_iterations
        self.theta = None
        self.loss_history = []
        
    def _sigmoid(self, z):
        z = np.clip(z, -500, 500)
        return 1 / (1 + np.exp(-z))
        
    def fit(self, X, y):
        m, n = X.shape
        X_b = np.c_[np.ones((m, 1)), X]
        self.theta = np.zeros(n + 1)
        
        for i in range(self.n_iter):
            z = X_b.dot(self.theta)
            h = self._sigmoid(z)
            eps = 1e-15
            loss = -np.mean(y * np.log(h + eps) + (1 - y) * np.log(1 - h + eps))
            self.loss_history.append(loss)
            
            gradient = X_b.T.dot(h - y) / m
            self.theta -= self.lr * gradient
            
        self.intercept = self.theta[0]
        self.coef = self.theta[1:]
        
    def predict_proba(self, X):
        X_b = np.c_[np.ones((X.shape[0], 1)), X]
        return self._sigmoid(X_b.dot(self.theta))
    
    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

log_reg = LogisticRegressionGD(learning_rate=0.1, n_iterations=10000)
log_reg.fit(X_train_b, y_train_b)
print(f"Intercept: {log_reg.intercept:.4f}, Coef: {log_reg.coef}")

plt.figure(figsize=(8, 4))
plt.plot(log_reg.loss_history, color='purple')
plt.title('Grafik Log Loss vs Iterasi')
plt.xlabel('Iterasi')
plt.ylabel('Log Loss')
plt.grid(True)
plt.show()

In [ ]:
def confusion_matrix_fn(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    return np.array([[tn, fp], [fn, tp]])

def precision_score(y_true, y_pred):
    cm = confusion_matrix_fn(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    return tp / (tp + fp) if (tp + fp) > 0 else 0.0

def recall_score(y_true, y_pred):
    cm = confusion_matrix_fn(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    return tp / (tp + fn) if (tp + fn) > 0 else 0.0

def f1_score_fn(y_true, y_pred):
    p = precision_score(y_true, y_pred)
    r = recall_score(y_true, y_pred)
    return 2 * (p * r) / (p + r) if (p + r) > 0 else 0.0

y_pred_b_test = log_reg.predict(X_test_b, threshold=0.5)
print("Confusion Matrix:\n", confusion_matrix_fn(y_test_b, y_pred_b_test))
print(f"Accuracy: {np.mean(y_test_b == y_pred_b_test):.4f}")
print(f"Precision: {precision_score(y_test_b, y_pred_b_test):.4f}")
print(f"Recall: {recall_score(y_test_b, y_pred_b_test):.4f}")
print(f"F1-Score: {f1_score_fn(y_test_b, y_pred_b_test):.4f}")

In [ ]:
# Eksperimen Threshold
thresholds = [0.3, 0.5, 0.7]
results = []
for th in thresholds:
    yp = log_reg.predict(X_test_b, threshold=th)
    cm = confusion_matrix_fn(y_test_b, yp)
    results.append({
        'Threshold': th,
        'Accuracy': np.mean(y_test_b == yp),
        'Precision': precision_score(y_test_b, yp),
        'Recall': recall_score(y_test_b, yp),
        'F1-score': f1_score_fn(y_test_b, yp),
        'CM': str(cm.tolist())
    })
display(pd.DataFrame(results))

# Prediksi Email Baru
new_emails = pd.DataFrame({
    'Email': ['E1', 'E2', 'E3', 'E4'],
    'JumlahLink': [1, 5, 9, 2],
    'KataPromo': [2, 6, 10, 11]
})
X_new = new_emails[['JumlahLink', 'KataPromo']].values
new_emails['Prob_Spam'] = log_reg.predict_proba(X_new)
new_emails['Pred_Spam'] = log_reg.predict(X_new, threshold=0.5)
display(new_emails)

## Bagian C – Pemahaman Konsep & Bonus Scikit-Learn

In [ ]:
from sklearn.linear_model import LinearRegression, LogisticRegression as SklearnLogisticRegression

sk_lr = LinearRegression()
sk_lr.fit(X_train_a, y_train_a)
print("Sklearn Linear Regression Intercept:", sk_lr.intercept_)
print("Sklearn Linear Regression Coef:", sk_lr.coef_)

sk_log = SklearnLogisticRegression(penalty=None, solver='lbfgs', max_iter=10000)
sk_log.fit(X_train_b, y_train_b)
print("Sklearn Logistic Regression Intercept:", sk_log.intercept_)
print("Sklearn Logistic Regression Coef:", sk_log.coef_)